# PyCAM-SIMA live-field Notebook

This Notebook starts one controller plus 24 CAM-SIMA MPI workers, advances the model one step at a time, and returns live NumPy fields to this kernel.

It works from a Derecho login-node Jupyter kernel: `model.start()` automatically submits a 24-rank PBS worker and waits for it to connect. Inside an existing compute allocation it launches locally. If `pycam_sima` is not installed in the selected kernel, run `%pip install -e /glade/work/ruitong/pycam-sima` once and restart the kernel.

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

from pycam_sima import NotebookSession

repo = Path("/glade/work/ruitong/pycam-sima")
case = repo / "reference/cases/FKESSLER_ne3pg3_gnu_24x50"
reference_run = Path(
    "/glade/derecho/scratch/ruitong/pycam-sima/"
    "FKESSLER_ne3pg3_gnu_24x50/FKESSLER_ne3pg3_gnu_24x50/run"
)
scratch = Path(os.environ.get("SCRATCH", "/glade/derecho/scratch/ruitong"))
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run_dir = scratch / "pycam-sima/notebook_trials" / stamp / "run"
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_run / "atm_in", run_dir / "atm_in")
print(run_dir)

## Start the MPI model

This cell returns only after all 24 ranks have completed CAM initialization. Re-running it first closes an older session. If that session already produced history files, run the setup cell again to create a fresh directory.

In [ ]:
if "model" in globals() and model.running:
    model.close()

existing_history = tuple(run_dir.glob("*.cam.h*.nc"))
if existing_history:
    raise RuntimeError("run the setup cell again to create a fresh run directory")

model = NotebookSession(
    repo / "configs/fkessler_ne3pg3.yaml",
    run_dir=run_dir,
    env_script=case / ".env_mach_specific.sh",
    python_executable=repo / ".venv/bin/python",
    log_path=run_dir / "mpi-worker.log",
)
model.start()
print(
    f"ready: mode={model.launch_mode_used}, job={model.job_id}, "
    f"ranks={model.ranks}, fields={len(model.field_names)}, step={model.current_step}"
)

## Inspect available fields

In [ ]:
model.field_names

## Read a field before stepping

`get_field` returns a copied rank-local NumPy array. Here `temperature` remains directly available to later Notebook cells.

In [ ]:
field_name = "air_temperature"
rank = 0
temperature = model.get_field(field_name, rank=rank)
stats = model.get_field_stats(field_name, rank=rank)
print(stats)
temperature

## Advance exactly one requested step and read the new value

In [ ]:
step = model.step()
temperature = model.get_field(field_name, rank=rank)
stats = model.get_field_stats(field_name, rank=rank)
print(f"step={step}, T[0,0]={temperature[0, 0]:.17g}")
print(stats)
temperature

## Continue step by step

Change `additional_steps` as needed. Each iteration obtains the field from the live MPI model, not from an NPZ file.

In [ ]:
additional_steps = 4
for _ in range(additional_steps):
    step = model.step()
    temperature = model.get_field(field_name, rank=rank)
    stats = model.get_field_stats(field_name, rank=rank)
    print(
        f"step={step} T[0,0]={temperature[0, 0]:.17g} "
        f"min={stats['min']:.17g} max={stats['max']:.17g}"
    )

## Optional: modify a live field

Running the next cell changes rank-zero CAM memory and intentionally breaks BFB. It is left commented out by default.

In [ ]:
# changed = model.get_field("air_temperature", rank=0)
# changed[0, 0] += 0.01
# model.set_field("air_temperature", changed, rank=0)
# model.step()

## Finalize

Always run this cell when finished so CAM finalizes and all MPI worker processes exit.

In [ ]:
model.close()
print(f"closed; output directory: {run_dir}")